# ***GÖREV 1***

In [ ]:
!wget  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-03 11:55:34--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  7.09MB/s    in 0.2s    

2026-10-03 11:55:35 (7.09 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [ ]:
with open("input.txt","r",encoding="utf-8") as f:
  text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [ ]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
stoi = {s : i for i,s in enumerate(chars)}
itos = {i : s for i,s in enumerate(chars)}
encode = lambda s : [stoi[c] for c in s]
decode = lambda l : ''.join([itos[i] for i in l])

print(encode("merhabalar"))
print(decode(encode("merhabalar")))

[51, 43, 56, 46, 39, 40, 39, 50, 39, 56]
merhabalar


In [ ]:
encode("Kal")

[23, 39, 50]

In [ ]:
decode([12,22,13,45])

'?JAg'

In [ ]:
import torch
data = torch.tensor(encode(text), dtype = torch.long) # torch.long kullanılyor çünkü bu karakterlerin tam sayı kimlikleri
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]



In [ ]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1 : block_size + 1]

for t in range(block_size):
  context = x[:t +1]
  target = y[t]
  print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [ ]:
torch.manual_seed(1234)
batch_size = 4
block_size = 8


def get_batch(split):

  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - block_size,(batch_size,))
  #torch.stack, aldığım parçaları aynı tensörde bir araya getiriyor.
  x = torch.stack([data[i : i + block_size] for i in ix])
  y = torch.stack([data[i+1: i +block_size + 1] for i in ix])
  return x,y

xb, yb = get_batch('train')
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

print("----")


for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t +1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target.tolist()}")

inputs:
torch.Size([4, 8])
tensor([[51, 59, 57, 58,  1, 39, 61, 39],
        [63,  6,  1, 61, 46, 53,  5, 57],
        [39,  1, 41, 46, 47, 50, 42, 10],
        [53, 59, 10,  0, 37, 53, 59,  1]])
targets:
torch.Size([4, 8])
tensor([[59, 57, 58,  1, 39, 61, 39, 63],
        [ 6,  1, 61, 46, 53,  5, 57,  1],
        [ 1, 41, 46, 47, 50, 42, 10,  0],
        [59, 10,  0, 37, 53, 59,  1, 57]])
----
when input is [51] the target: 59
when input is [51, 59] the target: 57
when input is [51, 59, 57] the target: 58
when input is [51, 59, 57, 58] the target: 1
when input is [51, 59, 57, 58, 1] the target: 39
when input is [51, 59, 57, 58, 1, 39] the target: 61
when input is [51, 59, 57, 58, 1, 39, 61] the target: 39
when input is [51, 59, 57, 58, 1, 39, 61, 39] the target: 63
when input is [63] the target: 6
when input is [63, 6] the target: 1
when input is [63, 6, 1] the target: 61
when input is [63, 6, 1, 61] the target: 46
when input is [63, 6, 1, 61, 46] the target: 53
when input is [63, 6, 

In [ ]:
print(xb)

tensor([[51, 59, 57, 58,  1, 39, 61, 39],
        [63,  6,  1, 61, 46, 53,  5, 57],
        [39,  1, 41, 46, 47, 50, 42, 10],
        [53, 59, 10,  0, 37, 53, 59,  1]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1234)



class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
       super().__init__()

       self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B*T, C), #Böylece bütün konumlardaki tahminleri alt alta koymuş oluyorum.
                targets.reshape(B*T))   #Hedefleri de aynı sırayla düzleştiriyorum
        return logits, loss   # (B, T, C)


    def generate(self,idx, max_new_tokens):

      for _ in range(max_new_tokens):

          logits, loss = self(idx)

          logits = logits[:, -1, :] # son metnin puanını alıyorum
          #İlk iki nokta bütün metinleri, eksi bir son konumu, son iki nokta ise bütün karakter seçeneklerinin puanlarını alıyor.
          probs = F.softmax(logits, dim = -1)
          #torch.multinomial ile bu olasılıklara göre bir token seçiyorum.
          idx_next = torch.multinomial(probs, num_samples=1) #her metin için bir token seçmemi sağlıyor.
# torch.cat satırında seçilen token’ı mevcut metnin sonuna ekliyorum. dim=1, birleştirmeyi metnin uzunluğu boyunca yapıyor.
          idx = torch.cat((idx,idx_next), dim = 1)

      return idx



m = BigramLanguageModel(vocab_size)
logits,loss = m(xb,yb)
print(logits.shape)
print(loss)



print(decode(m.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([4, 8, 65])
tensor(4.8650, grad_fn=<NllLossBackward0>)

UkBP-NBU& Y!lL ey-CofWUelZ?Jz.saHcpq!Uo's$VR&mP'xOXt3fTxwKb$!?$Kn?,aUs'ao3Jm:QcJwp
K!Uh:mEkwOssPcDIU


In [ ]:
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)


In [ ]:
for step in range(10000):
    xb, yb = get_batch("train")
    opt.zero_grad(set_to_none=True)
    logits, loss = m(xb, yb)
    loss.backward()
    opt.step()

    print(loss.item())

Görüntülenen çıkış son 5000 satıra kısaltıldı.
3.040830612182617
2.9296019077301025
2.82731032371521
2.925055980682373
2.606808662414551
2.70751690864563
2.8551158905029297
3.077026844024658
3.1835246086120605
2.7431480884552
3.104912757873535
2.531519651412964
2.8666090965270996
2.7937097549438477
2.776553153991699
2.9385485649108887
2.6643669605255127
2.917268991470337
2.6945152282714844
2.4476168155670166
2.4985191822052
2.926685333251953
2.7994842529296875
2.5135607719421387
2.5701494216918945
3.1874098777770996
2.6511270999908447
2.9742541313171387
2.7766976356506348
2.7796831130981445
2.7134997844696045
3.0020880699157715
2.9140079021453857
2.772258758544922
2.9477460384368896
2.8254759311676025
2.612806797027588
2.836456298828125
2.477597951889038
2.9091274738311768
2.7492620944976807
2.952166795730591
3.0147833824157715
2.7490038871765137
2.7591640949249268
2.7366786003112793
2.5797533988952637
3.1352081298828125
2.9882731437683105
3.005296230316162
2.7432775497436523
3.1120762

#***GÖREV 2***

In [ ]:
torch.manual_seed(1234)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b, :t +1] #(t,C)
    xbow[b,t] = torch.mean(xprev,0)

In [ ]:
x[0]

tensor([[-0.1117, -0.4966],
        [ 0.1631, -0.8817],
        [ 0.0539,  0.6684],
        [-0.0597, -0.4675],
        [-0.2153,  0.8840],
        [-0.7584, -0.3689],
        [-0.3424, -1.4020],
        [ 0.3206, -1.0219]])

In [ ]:
xbow[0]

tensor([[-0.1117, -0.4966],
        [ 0.0257, -0.6891],
        [ 0.0351, -0.2366],
        [ 0.0114, -0.2944],
        [-0.0339, -0.0587],
        [-0.1547, -0.1104],
        [-0.1815, -0.2949],
        [-0.1187, -0.3858]])

In [ ]:
torch.manual_seed(1234)
a = torch.ones(3,3)
a = a / torch.sum(a,1, keepdim = True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print("a=")
print(a)
print("----------")
print("b=")
print(b)
print("----------")
print("c=")
print(c)


a=
tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333]])
----------
b=
tensor([[5., 1.],
        [6., 5.],
        [6., 4.]])
----------
c=
tensor([[5.6667, 3.3333],
        [5.6667, 3.3333],
        [5.6667, 3.3333]])


In [ ]:
torch.tril(torch.ones(3,3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [ ]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [ ]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x #(B,T,C) @ (B,T,T) --> (B,T,C)
torch.allclose(xbow, xbow2)

True

In [ ]:
xbow, xbow2

(tensor([[[-1.1172e-01, -4.9659e-01],
          [ 2.5678e-02, -6.8914e-01],
          [ 3.5085e-02, -2.3663e-01],
          [ 1.1399e-02, -2.9435e-01],
          [-3.3931e-02, -5.8688e-02],
          [-1.5468e-01, -1.1038e-01],
          [-1.8150e-01, -2.9490e-01],
          [-1.1873e-01, -3.8577e-01]],
 
         [[ 7.9883e-01, -9.2288e-02],
          [ 4.6956e-02, -8.4732e-01],
          [ 1.2767e-01, -4.0159e-01],
          [-5.6613e-04, -4.7920e-01],
          [-3.4576e-02, -6.7525e-01],
          [ 7.9635e-03, -5.2167e-01],
          [-1.8243e-01, -3.4757e-01],
          [-2.4251e-01, -1.5216e-01]],
 
         [[-1.4949e+00,  8.8096e-01],
          [-1.3368e+00, -2.6517e-02],
          [-1.0804e+00, -1.1009e-01],
          [-1.3561e+00,  9.1412e-03],
          [-8.9729e-01,  8.8763e-03],
          [-8.0005e-01, -1.8539e-01],
          [-4.2278e-01, -3.0425e-01],
          [-2.1753e-01, -2.4620e-01]],
 
         [[ 1.5985e+00, -4.6861e-02],
          [ 3.5753e-02, -1.0306e+00],
   

In [ ]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei,dim= 1)
xbow3 = wei @ x
torch.allclose(xbow,xbow3)

True

In [ ]:
torch.manual_seed(1234)
B,T,C = 4,8,32
x = torch.randn(B,T,C)
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
wei =  q @ k.transpose(-2, -1)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
v = value(x)
out = wei @ v
out.shape

torch.Size([4, 8, 16])

In [ ]:
scores = q @ k.transpose(-2, -1)
scores = scores.masked_fill(tril == 0, float("-inf"))

without_scaling = F.softmax(scores, dim=-1)
with_scaling = F.softmax(scores / (head_size ** 0.5), dim=-1)

print(torch.stack([
    without_scaling[0, 2],
    with_scaling[0, 2]
], dim=1))

tensor([[0.2088, 0.3195],
        [0.7186, 0.4352],
        [0.0726, 0.2453],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 0.0000]], grad_fn=<StackBackward0>)


# ***GÖREV 3***

In [ ]:
B, T, C = 4, 8, 32
head_size = 16

# 1. Örnek token vektörleri
x = torch.randn(B, T, C)

# 2. Her token'a bulunduğu konumun bilgisini ekle
position_embedding_table = nn.Embedding(T, C)
positions = torch.arange(T)
pos_emb = position_embedding_table(positions)  # (T, C)

x = x + pos_emb                               # (B, T, C)

# 3. Tek head için query, key ve value
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)    # (B, T, 16)
q = query(x)  # (B, T, 16)
v = value(x)  # (B, T, 16)

# 4. Token'lar arasındaki eşleşme puanları
scores = q @ k.transpose(-2, -1)  # (B, T, T)

# 5. Gelecekteki konumları kapatan maske
tril = torch.tril(torch.ones(T, T))

# 6. Karekök bölmesi OLMADAN attention ağırlıkları
scores_unscaled = scores.masked_fill(
    tril == 0, float("-inf")
)
wei_unscaled = F.softmax(scores_unscaled, dim=-1)

# 7. Karekök bölmesi YAPARAK attention ağırlıkları
scores_scaled = scores / (head_size ** 0.5)
scores_scaled = scores_scaled.masked_fill(
    tril == 0, float("-inf")
)
wei = F.softmax(scores_scaled, dim=-1)

# 8. Value vektörlerini ağırlıklarla birleştir
out = wei @ v  # (B, T, 16)

print("Çıktı şekli:", out.shape)

# 9. İlk dizinin attention matrisi
print("\nİlk dizinin wei matrisi:")
print(wei[0])


b, t = 0, 2

print("\nÜçüncü token'ın ağırlıkları:")
print(wei[b, t])

for j in range(t + 1):
    print(
        f"3. token, {j + 1}. token'ın V vektörünü "
        f"%{100 * wei[b, t, j].item():.2f} ağırlıkla kullanıyor."
    )

print("\nToken | Bölmeden | Bölerek")

for j in range(T):
    print(
        f"{j + 1:5d} | "
        f"{wei_unscaled[b, t, j].item():8.4f} | "
        f"{wei[b, t, j].item():7.4f}"
    )

Çıktı şekli: torch.Size([4, 8, 16])

İlk dizinin wei matrisi:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6415, 0.3585, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1948, 0.5323, 0.2730, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1692, 0.2795, 0.2961, 0.2551, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5062, 0.1000, 0.0545, 0.0941, 0.2453, 0.0000, 0.0000, 0.0000],
        [0.1042, 0.2384, 0.0870, 0.2811, 0.1074, 0.1819, 0.0000, 0.0000],
        [0.1135, 0.1322, 0.1099, 0.0717, 0.1538, 0.2817, 0.1372, 0.0000],
        [0.1057, 0.1156, 0.0940, 0.2171, 0.1215, 0.1348, 0.1296, 0.0817]],
       grad_fn=<SelectBackward0>)

Üçüncü token'ın ağırlıkları:
tensor([0.1948, 0.5323, 0.2730, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       grad_fn=<SelectBackward0>)
3. token, 1. token'ın V vektörünü %19.48 ağırlıkla kullanıyor.
3. token, 2. token'ın V vektörünü %53.23 ağırlıkla kullanıyor.
3. token, 3. token'ın V vektörünü %27.30 ağırlık

# ***GÖREV 4***

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [ ]:
k.var()

tensor(0.8799)

In [ ]:
q.var()

tensor(0.9552)

In [ ]:
wei.var()

tensor(0.7751)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1)

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [ ]:
class LayerNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):

    xmean = x.mean(1, keepdim=True)
    xvar = x.var(1, keepdim=True)
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])

In [ ]:
x[:,0].mean(), x[:,0].std()

(tensor(0.1469), tensor(0.8803))

In [ ]:
x[0,:].mean(), x[0,:].std()

(tensor(-9.5367e-09), tensor(1.0000))

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0


torch.manual_seed(1337)


with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):

        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):


    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')


    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5090, val loss 2.5058
step 300: train loss 2.4194, val loss 2.4334
step 400: train loss 2.3501, val loss 2.3568
step 500: train loss 2.2963, val loss 2.3129
step 600: train loss 2.2410, val loss 2.2501
step 700: train loss 2.2057, val loss 2.2191
step 800: train loss 2.1633, val loss 2.1860
step 900: train loss 2.1242, val loss 2.1498
step 1000: train loss 2.1027, val loss 2.1298
step 1100: train loss 2.0692, val loss 2.1183
step 1200: train loss 2.0386, val loss 2.0797
step 1300: train loss 2.0276, val loss 2.0652
step 1400: train loss 1.9925, val loss 2.0370
step 1500: train loss 1.9702, val loss 2.0302
step 1600: train loss 1.9645, val loss 2.0487
step 1700: train loss 1.9421, val loss 2.0143
step 1800: train loss 1.9091, val loss 1.9953
step 1900: train loss 1.9085, val loss 1.9874
step 2000: train loss 1.8861, val loss 1.9957
step 2100: train loss 1.